# Market Basket Analysis

Generate overall and segment-level association rules, top products, and cross-sell opportunities.

This notebook is a cleaned and renamed version of the original analysis notebook.

## Notebook Guide
This notebook mines co-purchase patterns and turns them into segment-specific product recommendations for cross-sell and basket-building opportunities.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'dataset'
GENERATED_DIR = PROJECT_ROOT / 'generated'
FIGURES_DIR = PROJECT_ROOT / 'figures'

GENERATED_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset dir: {DATA_DIR}')
print(f'Generated dir: {GENERATED_DIR}')
print(f'Figures dir: {FIGURES_DIR}')


Project root: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis
Dataset dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/dataset
Generated dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/generated
Figures dir: /Users/ponnimuthukumarasamy/Downloads/Prosol Dataset Complete/Sales-Data-Analysis/figures


In [2]:
"""
Customer Segmentation Analysis - Part 3: Market Basket & Recommendations
========================================================================

This notebook provides:
1. Market basket analysis and association rules
2. Segment-specific product recommendations
3. Cross-selling and upselling opportunities
4. Final business recommendations
"""

'\nCustomer Segmentation Analysis - Part 3: Market Basket & Recommendations\n========================================================================\n\nThis notebook provides:\n1. Market basket analysis and association rules\n2. Segment-specific product recommendations\n3. Cross-selling and upselling opportunities\n4. Final business recommendations\n'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Load data
print("Loading data...")
df_transactions = pd.read_csv(DATA_DIR / 'TOKEN.csv').merge(
    pd.read_csv(DATA_DIR / 'VENTE.csv'), on='CD_TICKET_UNIQUE'
).merge(
    pd.read_csv(DATA_DIR / 'ARTICLE.csv'), on='ID_ARTICLE', how='left'
)

customer_segments = pd.read_csv(GENERATED_DIR / 'customers_with_segments.csv')

print(f"✓ Transaction data loaded: {df_transactions.shape}")
print(f"✓ Customer segments loaded: {customer_segments.shape}")
customer_segments

Loading data...
✓ Transaction data loaded: (1305191, 18)
✓ Customer segments loaded: (121112, 28)


,NO_TOKEN,DT_VENTE_first_purchase,DT_VENTE_last_purchase,DT_VENTE_shopping_days,CD_TICKET_UNIQUE_total_transactions,MT_TTC_NET_total_spend,MT_TTC_NET_avg_transaction_value,LB_METIER_dept_diversity,LB_FAMILLE_family_diversity,TOTAL_QTY_total_items_bought,...,avg_items_per_basket,avg_basket_value,unit_weight_ratio,promo_percentage,national_vs_store_promo_ratio,store_loyalty_score,product_diversity_score,avg_days_between_visits,combinaison,Cluster
0,00000ED6FFB345A27CF94114C9819546E9E65D333D0D10...,2025-12-09,2025-12-10,2,2,26.59,1.564118,3,10,19.003,...,9.50150,13.2950,3.553380,18.128717,3.445,0.333333,30,0.5,0.506862,0
1,00000F282C0A8BE112B1CB5EBCEB61673125F374902663...,2025-12-05,2025-12-23,4,4,71.55,3.110870,4,17,32.997,...,8.24925,17.8875,1.829075,0.000000,0.000,0.250000,68,4.5,0.220365,2
2,00005F300675C1E7BA7847B7759248F908230E87BC1EE3...,2025-12-21,2025-12-21,1,1,37.04,6.173333,3,5,8.460,...,8.46000,37.0400,4.468208,0.000000,0.000,0.333333,15,0.0,-0.343284,0
3,0001421783EEDB9B602293092CB337D3A69B84C3B42712...,2025-12-18,2025-12-22,2,2,49.39,4.490000,4,9,12.586,...,6.29300,24.6950,5.705824,0.000000,0.000,0.250000,36,2.0,-0.087035,0
4,0001B89407D2DCFD6749D4C8A1B0A164E010BA61B1C68D...,2025-12-22,2025-12-22,1,1,21.98,7.326667,2,2,4.000,...,4.00000,21.9800,4.000000,0.000000,0.000,0.500000,4,0.0,-0.583499,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121107,FFFDCB914FD6ECFE1B5CBF1177878C5D0EBFDA6F7C8701...,2025-12-02,2025-12-02,1,1,2.47,2.470000,1,1,1.196,...,1.19600,2.4700,0.836120,0.000000,0.000,1.000000,1,0.0,0.227773,3
121108,FFFEA739305DA4E455CD858CAA7F160073E4E075188425...,2025-12-04,2025-12-04,1,1,1.19,1.190000,1,1,1.000,...,1.00000,1.1900,1.000000,0.000000,0.000,1.000000,1,0.0,0.106418,3
121109,FFFF9B5B54BBB8090D7B01BD1CDB99C26745E8379E863B...,2025-12-04,2025-12-09,2,2,18.81,2.687143,2,3,7.295,...,3.64750,9.4050,5.405405,0.000000,0.000,0.500000,6,2.5,0.377310,0
121110,FFFFC67A3EF5810BB38FA9371911E3DBC07886046BB0A8...,2025-12-01,2025-12-01,1,1,12.46,3.115000,2,2,4.000,...,4.00000,12.4600,4.000000,0.000000,0.000,0.500000,4,0.0,0.404864,0


## PART 7: MARKET BASKET ANALYSIS
Build the transaction baskets needed for association-rule mining.


In [5]:
# ============================================================================
# PART 7: MARKET BASKET ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("PART 7: MARKET BASKET ANALYSIS")
print("=" * 80)


PART 7: MARKET BASKET ANALYSIS


## STEP 7.1: Prepare Transaction Baskets
Transform line-level sales into product sets that represent each basket.


In [6]:
# ============================================================================
# STEP 7.1: Prepare Transaction Baskets
# ============================================================================

print("\n[STEP 7.1] Preparing transaction baskets...")

# Create baskets at family level (more manageable than individual products)
baskets = df_transactions.groupby(['CD_TICKET_UNIQUE', 'LB_FAMILLE'])['QT_UVC'].sum().reset_index()
baskets = baskets[baskets['LB_FAMILLE'].notna()]

# Create basket format for association rules
basket_items = baskets.groupby('CD_TICKET_UNIQUE')['LB_FAMILLE'].apply(list).reset_index()

print(f"✓ Created {len(basket_items)} baskets")
print(f"  Average items per basket: {baskets.groupby('CD_TICKET_UNIQUE').size().mean():.2f}")

# Sample of baskets
print("\nSample baskets:")
for i in range(min(5, len(basket_items))):
    print(f"  Basket {i+1}: {basket_items.iloc[i]['LB_FAMILLE'][:5]}")



[STEP 7.1] Preparing transaction baskets...
✓ Created 174007 baskets
  Average items per basket: 6.32

Sample baskets:
  Basket 1: ['Agrumes', 'Banane', 'Prêt à déguster', 'Pâte fraîche']
  Basket 2: ['Agrumes', 'Avocat', 'Banane', 'Courge', 'Courgette']
  Basket 3: ['Plat familial']
  Basket 4: ['Banane', 'Pomme']
  Basket 5: ['Oeufs', 'Poissons']


## STEP 7.2: Generate Association Rules - Overall
Mine global product affinities across the full customer base.


In [7]:
# ============================================================================
# STEP 7.2: Generate Association Rules - Overall
# ============================================================================

print("\n[STEP 7.2] Generating association rules (overall)...")

# Convert to one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(basket_items['LB_FAMILLE']).transform(basket_items['LB_FAMILLE'])
basket_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"✓ One-hot encoded baskets: {basket_encoded.shape}")

# Find frequent itemsets
print("  Finding frequent itemsets...")
frequent_itemsets = apriori(basket_encoded, min_support=0.01, use_colnames=True)
print(f"✓ Found {len(frequent_itemsets)} frequent itemsets")

# Generate association rules
print("  Generating association rules...")
if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
    rules = rules.sort_values('lift', ascending=False)
    
    print(f"✓ Generated {len(rules)} association rules")
    
    # Display top rules
    print("\nTop 10 Association Rules (by Lift):")
    print("=" * 120)
    top_rules = rules.head(10)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    
    for idx, row in top_rules.iterrows():
        ant = list(row['antecedents'])[0] if len(row['antecedents']) == 1 else row['antecedents']
        cons = list(row['consequents'])[0] if len(row['consequents']) == 1 else row['consequents']
        print(f"\n{ant}")
        print(f"  → {cons}")
        print(f"  Support: {row['support']:.3f} | Confidence: {row['confidence']:.3f} | Lift: {row['lift']:.2f}")
    
    # Save rules
    rules.to_csv(GENERATED_DIR / 'association_rules_overall.csv', index=False)
    print("\n✓ Association rules saved to 'association_rules_overall.csv'")
else:
    print("⚠ No frequent itemsets found. Consider lowering min_support threshold.")
    rules = pd.DataFrame()



[STEP 7.2] Generating association rules (overall)...
✓ One-hot encoded baskets: (174007, 120)
  Finding frequent itemsets...
✓ Found 2226 frequent itemsets
  Generating association rules...
✓ Generated 12222 association rules

Top 10 Association Rules (by Lift):

frozenset({'Poireau', 'Carotte'})
  → frozenset({'Navet et Rave', 'Agrumes'})
  Support: 0.011 | Confidence: 0.240 | Lift: 8.36

frozenset({'Navet et Rave', 'Agrumes'})
  → frozenset({'Poireau', 'Carotte'})
  Support: 0.011 | Confidence: 0.370 | Lift: 8.36

Navet et Rave
  → frozenset({'Poireau', 'Agrumes', 'Carotte'})
  Support: 0.011 | Confidence: 0.238 | Lift: 7.58

frozenset({'Poireau', 'Agrumes', 'Carotte'})
  → Navet et Rave
  Support: 0.011 | Confidence: 0.339 | Lift: 7.58

Navet et Rave
  → frozenset({'Poireau', 'Carotte'})
  Support: 0.015 | Confidence: 0.329 | Lift: 7.43

frozenset({'Poireau', 'Carotte'})
  → Navet et Rave
  Support: 0.015 | Confidence: 0.332 | Lift: 7.43

frozenset({'Poireau', 'Agrumes'})
  → froze

## STEP 7.3: Segment-Specific Association Rules
Repeat the analysis at the segment level to uncover more targeted product links.


In [14]:
# ============================================================================
# STEP 7.3: Segment-Specific Association Rules
# ============================================================================

print("\n[STEP 7.3] Generating segment-specific association rules...")

# Merge segments with transactions
df_trans_segments = df_transactions.merge(
    customer_segments[['NO_TOKEN', 'Cluster']], 
    left_on='NO_TOKEN_CB', 
    right_on = "NO_TOKEN",
    how='left'
)

# Generate rules for each segment
segment_rules_dict = {}

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    print(f"\n  Analyzing Segment {cluster_id}...")
    
    # Get transactions for this segment
    segment_trans = df_trans_segments[df_trans_segments['Cluster'] == cluster_id]
    
    # Create baskets
    segment_baskets = segment_trans.groupby(['CD_TICKET_UNIQUE', 'LB_FAMILLE'])['QT_UVC'].sum().reset_index()
    segment_baskets = segment_baskets[segment_baskets['LB_FAMILLE'].notna()]
    
    if len(segment_baskets) < 10:
        print(f"    ⚠ Insufficient data for Segment {cluster_id}")
        continue
    
    segment_basket_items = segment_baskets.groupby('CD_TICKET_UNIQUE')['LB_FAMILLE'].apply(list).reset_index()
    
    # Encode and find patterns
    try:
        te_seg = TransactionEncoder()
        te_seg_ary = te_seg.fit(segment_basket_items['LB_FAMILLE']).transform(segment_basket_items['LB_FAMILLE'])
        basket_seg_encoded = pd.DataFrame(te_seg_ary, columns=te_seg.columns_)
        
        freq_items_seg = apriori(basket_seg_encoded, min_support=0.02, use_colnames=True)
        
        if len(freq_items_seg) > 0:
            rules_seg = association_rules(freq_items_seg, metric="lift", min_threshold=1.0)
            rules_seg = rules_seg.sort_values('lift', ascending=False)
            segment_rules_dict[cluster_id] = rules_seg
            
            print(f"    ✓ Found {len(rules_seg)} rules for Segment {cluster_id}")
            
            # Save segment rules
            rules_seg.to_csv(GENERATED_DIR / f'association_rules_segment_{cluster_id}.csv', index=False)
        else:
            print(f"    ⚠ No rules found for Segment {cluster_id}")
    except Exception as e:
        print(f"    ⚠ Error processing Segment {cluster_id}: {str(e)}")



[STEP 7.3] Generating segment-specific association rules...

  Analyzing Segment 0...
    ✓ Found 746 rules for Segment 0

  Analyzing Segment 1...
    ✓ Found 5384 rules for Segment 1

  Analyzing Segment 2...
    ✓ Found 14524 rules for Segment 2

  Analyzing Segment 3...
    ✓ Found 300 rules for Segment 3


## PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT
Convert the association rules into practical recommendations for each segment.


In [15]:
# ============================================================================
# PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT
# ============================================================================

print("\n" + "=" * 80)
print("PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT")
print("=" * 80)



PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT


## STEP 8.1: Top Products by Segment
Identify the most relevant products within each customer group.


In [16]:
# ============================================================================
# STEP 8.1: Top Products by Segment
# ============================================================================

print("\n[STEP 8.1] Identifying top products by segment...")

# Get top product families by segment
top_products_by_segment = df_trans_segments.groupby(['Cluster', 'LB_FAMILLE']).agg({
    'QT_UVC': 'sum',
    'MT_TTC_NET': 'sum',
    'CD_TICKET_UNIQUE': 'nunique'
}).reset_index()

top_products_by_segment.columns = ['Cluster', 'Product_Family', 'Total_Quantity', 'Total_Revenue', 'Unique_Transactions']
top_products_by_segment = top_products_by_segment.sort_values(['Cluster', 'Total_Revenue'], ascending=[True, False])

print("\nTop 5 Product Families by Segment:")
print("=" * 100)

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    segment_products = top_products_by_segment[top_products_by_segment['Cluster'] == cluster_id].head(5)
    print(f"\n📊 Segment {cluster_id}:")
    for idx, row in segment_products.iterrows():
        print(f"  {row['Product_Family']}: €{row['Total_Revenue']:,.0f} revenue | {row['Unique_Transactions']:,} transactions")

# Save
top_products_by_segment.to_csv(GENERATED_DIR / 'top_products_by_segment.csv', index=False)
print("\n✓ Top products saved to 'top_products_by_segment.csv'")



[STEP 8.1] Identifying top products by segment...

Top 5 Product Families by Segment:

📊 Segment 0:
  Agrumes: €184,120 revenue | 36,092 transactions
  Poissons: €114,955 revenue | 11,444 transactions
  Crustacés: €73,647 revenue | 8,185 transactions
  Coquillages: €59,468 revenue | 4,298 transactions
  Fruits Exotiques: €56,815 revenue | 11,999 transactions

📊 Segment 1:
  Agrumes: €23,261 revenue | 4,511 transactions
  Poissons: €20,667 revenue | 1,674 transactions
  Fromage à consommer chaud: €10,853 revenue | 685 transactions
  Crustacés: €7,594 revenue | 905 transactions
  Dessert: €6,675 revenue | 1,336 transactions

📊 Segment 2:
  Agrumes: €160,858 revenue | 26,019 transactions
  Poissons: €130,759 revenue | 10,287 transactions
  Crustacés: €76,787 revenue | 6,957 transactions
  Coquillages: €69,264 revenue | 4,010 transactions
  Poisson fumé: €62,097 revenue | 5,512 transactions

📊 Segment 3:
  Agrumes: €75,860 revenue | 13,145 transactions
  Légumes Exotiques: €32,121 revenue

## STEP 8.2: Cross-Selling Opportunities
Highlight the pairings most likely to increase basket size.


In [17]:
# ============================================================================
# STEP 8.2: Cross-Selling Opportunities
# ============================================================================

print("\n[STEP 8.2] Identifying cross-selling opportunities...")

# For each segment, find products frequently bought together
cross_sell_opportunities = []

for cluster_id, rules_df in segment_rules_dict.items():
    if len(rules_df) > 0:
        # Get top cross-sell rules
        top_cross_sell = rules_df.nlargest(5, 'lift')
        
        for idx, rule in top_cross_sell.iterrows():
            cross_sell_opportunities.append({
                'Segment': cluster_id,
                'If_Customer_Buys': list(rule['antecedents']),
                'Recommend': list(rule['consequents']),
                'Confidence': rule['confidence'],
                'Lift': rule['lift']
            })

cross_sell_df = pd.DataFrame(cross_sell_opportunities)

if len(cross_sell_df) > 0:
    print("\nTop Cross-Selling Opportunities by Segment:")
    print("=" * 100)
    for cluster_id in cross_sell_df['Segment'].unique():
        segment_opps = cross_sell_df[cross_sell_df['Segment'] == cluster_id]
        print(f"\n🎯 Segment {cluster_id}:")
        for idx, row in segment_opps.iterrows():
            print(f"  If buys: {row['If_Customer_Buys']}")
            print(f"    → Recommend: {row['Recommend']} (Lift: {row['Lift']:.2f}, Confidence: {row['Confidence']:.1%})")
    
    cross_sell_df.to_csv(GENERATED_DIR / 'cross_sell_opportunities.csv', index=False)
    print("\n✓ Cross-sell opportunities saved to 'cross_sell_opportunities.csv'")
else:
    print("⚠ No cross-sell opportunities identified")



[STEP 8.2] Identifying cross-selling opportunities...

Top Cross-Selling Opportunities by Segment:

🎯 Segment 0:
  If buys: ['Navet et Rave']
    → Recommend: ['Carotte'] (Lift: 4.39, Confidence: 63.3%)
  If buys: ['Carotte']
    → Recommend: ['Navet et Rave'] (Lift: 4.39, Confidence: 16.1%)
  If buys: ['Agrumes', 'Carotte']
    → Recommend: ['Poireau'] (Lift: 3.85, Confidence: 27.5%)
  If buys: ['Poireau']
    → Recommend: ['Agrumes', 'Carotte'] (Lift: 3.85, Confidence: 32.6%)
  If buys: ['Poireau', 'Agrumes']
    → Recommend: ['Carotte'] (Lift: 3.71, Confidence: 53.4%)

🎯 Segment 1:
  If buys: ['Courgette', 'Condiment']
    → Recommend: ['Banane', 'Carotte'] (Lift: 4.73, Confidence: 38.7%)
  If buys: ['Banane', 'Carotte']
    → Recommend: ['Courgette', 'Condiment'] (Lift: 4.73, Confidence: 24.5%)
  If buys: ['Courgette']
    → Recommend: ['Banane', 'Condiment', 'Carotte'] (Lift: 4.65, Confidence: 19.4%)
  If buys: ['Banane', 'Condiment', 'Carotte']
    → Recommend: ['Courgette'] (Li

## FINAL OUTPUT SUMMARY
Collect the key files and results generated by the market basket workflow.


In [18]:
# ============================================================================
# FINAL OUTPUT SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE!")
print("=" * 100)

print("\n📊 Generated Outputs:")
outputs = [
    "1. customer_features_prepared.csv - Customer features for clustering",
    "2. customers_with_segments.csv - All customers with assigned segments",
    "3. segment_profiles.csv - Statistical profiles of each segment",
    "4. segment_business_summary.csv - Business metrics by segment",
    "5. association_rules_overall.csv - Overall association rules",
    "6. association_rules_segment_X.csv - Segment-specific rules",
    "7. top_products_by_segment.csv - Best-selling products per segment",
    "8. cross_sell_opportunities.csv - Cross-selling recommendations",
    "9. store_segment_distribution.csv - Segment mix by store",
    "10. store_performance_by_segment.csv - Store-segment performance",
    "11. executive_summary.txt - Executive summary report",
    "12. Various visualization PNG files"
]

for output in outputs:
    print(f"  {output}")

print("\n🎯 Ready for Business Implementation!")
print("=" * 100)
























ANALYSIS COMPLETE!

📊 Generated Outputs:
  1. customer_features_prepared.csv - Customer features for clustering
  2. customers_with_segments.csv - All customers with assigned segments
  3. segment_profiles.csv - Statistical profiles of each segment
  4. segment_business_summary.csv - Business metrics by segment
  5. association_rules_overall.csv - Overall association rules
  6. association_rules_segment_X.csv - Segment-specific rules
  7. top_products_by_segment.csv - Best-selling products per segment
  8. cross_sell_opportunities.csv - Cross-selling recommendations
  9. store_segment_distribution.csv - Segment mix by store
  10. store_performance_by_segment.csv - Store-segment performance
  11. executive_summary.txt - Executive summary report
  12. Various visualization PNG files

🎯 Ready for Business Implementation!


## Results Summary

- Built basket-level transaction sets and mined association rules across the full customer base and within each segment.
- Identified the top product families by segment and highlighted cross-selling opportunities that align with segment behavior.
- Finished with a consolidated output summary of the recommendation artifacts created by the notebook.
- Main outputs: the segment-specific association-rule files and recommendation summaries saved under `generated/`.
